# V1-09: 기업 식별 `NOT_FOUND` 점수 cutoff 비교

08번에서 채택한 ‘명확한 정확 일치만 자동 확정’ 정책은 안전하지만, 존재하지 않는 기업명에도 유사 후보를 보여 주고 확인을 요청한다. 이번에는 문자열 유사도 최상위 점수가 낮으면 `NOT_FOUND`로 처리하는 cutoff가 이 문제를 줄이는지 비교한다.

## 비교 대상과 평가 기준

- **A. cutoff 없음**: 후보가 있으면 `NEEDS_CONFIRMATION`, 후보가 없을 때만 `NOT_FOUND`
- **B. cutoff 75점**: 문자열 유사 후보의 최고 점수가 75점 미만이면 `NOT_FOUND`, 그 외에는 `NEEDS_CONFIRMATION`

75점은 07·08번 개발 사례에서 가상기업 최고점(70점)과 한 글자 오타의 최고점(75점) 사이에 놓인 탐색용 후보일 뿐, 사전 확정된 운영 기준이 아니다. Not-found Recall, False Not-found Rate, Clarification Rate, p50/p95 지연시간을 비교한다. LLM은 사용하지 않는다.

In [1]:
import io
from math import ceil
import os
import re
import sys
import time
import xml.etree.ElementTree as ET
import zipfile
from pathlib import Path
from statistics import median

from rapidfuzz import fuzz, process
import rapidfuzz

In [2]:
current_dir = Path.cwd().resolve()
if current_dir.name == 'v1':
    project_root = current_dir.parents[1]
elif current_dir.name == 'notebooks':
    project_root = current_dir.parent
else:
    project_root = current_dir

env_path = project_root / '.env'
if not env_path.exists():
    raise FileNotFoundError(f'{env_path} 파일을 먼저 만들어 주세요.')

for raw_line in env_path.read_text(encoding='utf-8').splitlines():
    line = raw_line.strip()
    if line and not line.startswith('#') and '=' in line:
        key, value = line.split('=', 1)
        os.environ.setdefault(key.strip(), value.strip().strip(chr(39) + chr(34)))

api_key = os.environ.get('OPENDART_API_KEY')
if not api_key:
    raise ValueError('.env에 OPENDART_API_KEY를 설정해 주세요.')

In [3]:
sys.path.insert(0, str(project_root / 'src'))

from dart.client import CORP_CODE_URL, _get_bytes

print(f'RapidFuzz 버전: {rapidfuzz.__version__}')

RapidFuzz 버전: 3.14.5


## 1. OpenDART 기업명 인덱스

08번과 동일하게 상장·비상장 법인을 모두 후보로 유지한다. 상장 여부는 결과에 표시할 메타데이터이며 후보 제거 조건이 아니다.

In [4]:
def load_corp_records(api_key):
    zip_bytes = _get_bytes(CORP_CODE_URL, {'crtfc_key': api_key})
    zip_stream = io.BytesIO(zip_bytes)
    if not zipfile.is_zipfile(zip_stream):
        error_text = zip_bytes.decode('utf-8', errors='replace')
        raise RuntimeError(f'기업코드 응답이 ZIP이 아닙니다: {error_text}')
    with zipfile.ZipFile(zip_stream) as archive:
        xml_bytes = archive.read(archive.namelist()[0])

    root = ET.fromstring(xml_bytes)
    return [
        {child.tag: child.text or '' for child in item}
        for item in root.findall('list')
    ]

corp_records = load_corp_records(api_key)
print(f'전체 법인 레코드 수: {len(corp_records):,}')

전체 법인 레코드 수: 118,831


In [5]:
def normalize_name(value):
    return re.sub(r'[^0-9A-Z가-힣]', '', value.upper())

def build_name_index(records):
    index = {}
    for record in records:
        for name in {record['corp_name'], record['corp_eng_name']}:
            key = normalize_name(name)
            if key:
                index.setdefault(key, {})[record['corp_code']] = record
    return {key: list(records_by_code.values()) for key, records_by_code in index.items()}

name_index = build_name_index(corp_records)
name_keys = list(name_index)
print(f'정규화 이름 키 수: {len(name_keys):,}')

정규화 이름 키 수: 218,777


## 2. 대상 존재·부재 사례

대상 존재 사례는 공식명, 오타, 부분 표현, 별칭을 포함한다. 대상 부재 사례는 현재 기업명 인덱스에 정확 일치하지 않는 synthetic no-match다. 이는 실제 사용자 오입력의 모든 유형을 대표하지 않으며, cutoff가 낮은 유사도 후보를 거르는지 확인하는 최소 기준선이다.

In [6]:
TEST_CASES = [
    {'유형': '정확 법인명', '질문 기업명': '삼성전자', '대상 존재': True},
    {'유형': '영문 대소문자', '질문 기업명': 'naver', '대상 존재': True},
    {'유형': '한 글자 오타', '질문 기업명': '삼성전쟈', '대상 존재': True},
    {'유형': '부분 표현', '질문 기업명': '하이닉스', '대상 존재': True},
    {'유형': '짧은 약칭·동명', '질문 기업명': '삼전', '대상 존재': True},
    {'유형': '한글 별칭', '질문 기업명': '네이버', '대상 존재': True},
    {'유형': 'synthetic no-match', '질문 기업명': '핀사이트가상법인20260903가', '대상 존재': False},
    {'유형': 'synthetic no-match', '질문 기업명': '핀사이트가상법인20260903나', '대상 존재': False},
    {'유형': 'synthetic no-match', '질문 기업명': '핀사이트가상법인20260903다', '대상 존재': False},
    {'유형': 'synthetic no-match', '질문 기업명': '핀사이트가상법인20260903라', '대상 존재': False},
]

for case in TEST_CASES:
    if not case['대상 존재']:
        assert normalize_name(case['질문 기업명']) not in name_index

TEST_CASES

[{'유형': '정확 법인명', '질문 기업명': '삼성전자', '대상 존재': True},
 {'유형': '영문 대소문자', '질문 기업명': 'naver', '대상 존재': True},
 {'유형': '한 글자 오타', '질문 기업명': '삼성전쟈', '대상 존재': True},
 {'유형': '부분 표현', '질문 기업명': '하이닉스', '대상 존재': True},
 {'유형': '짧은 약칭·동명', '질문 기업명': '삼전', '대상 존재': True},
 {'유형': '한글 별칭', '질문 기업명': '네이버', '대상 존재': True},
 {'유형': 'synthetic no-match', '질문 기업명': '핀사이트가상법인20260903가', '대상 존재': False},
 {'유형': 'synthetic no-match', '질문 기업명': '핀사이트가상법인20260903나', '대상 존재': False},
 {'유형': 'synthetic no-match', '질문 기업명': '핀사이트가상법인20260903다', '대상 존재': False},
 {'유형': 'synthetic no-match', '질문 기업명': '핀사이트가상법인20260903라', '대상 존재': False}]

## 3. 후보와 `NOT_FOUND` 정책

정확 일치 하나는 두 정책에서 모두 `RESOLVED`다. 정확 일치가 여러 개면 점수와 무관하게 `NEEDS_CONFIRMATION`이다. 차이는 유사 검색으로 나온 후보의 최고 점수가 낮을 때다.

In [7]:
FUZZY_KEY_LIMIT = 20
RETURN_K = 3
CANDIDATE_SCORE_CUTOFF = 75.0

def rank_records(records, score=100.0):
    return [
        (record, score)
        for record in sorted(
            records,
            key=lambda record: (bool(record['stock_code'].strip()), record['modify_date']),
            reverse=True,
        )
    ]

def find_candidates(query):
    exact_records = name_index.get(normalize_name(query), [])
    if exact_records:
        return rank_records(exact_records)[:RETURN_K], 'exact'

    scored_keys = process.extract(
        normalize_name(query), name_keys, scorer=fuzz.ratio, limit=FUZZY_KEY_LIMIT
    )
    best_by_corp_code = {}
    for key, score, _ in scored_keys:
        for record in name_index[key]:
            current = best_by_corp_code.get(record['corp_code'])
            if current is None or score > current[1]:
                best_by_corp_code[record['corp_code']] = (record, score)

    ranked = sorted(
        best_by_corp_code.values(),
        key=lambda item: (item[1], bool(item[0]['stock_code'].strip()), item[0]['modify_date']),
        reverse=True,
    )
    return ranked[:RETURN_K], 'fuzzy'

def no_cutoff_policy(candidates, source):
    if source == 'exact' and len(candidates) == 1:
        return 'RESOLVED'
    return 'NEEDS_CONFIRMATION' if candidates else 'NOT_FOUND'

def cutoff_policy(candidates, source):
    if source == 'exact' and len(candidates) == 1:
        return 'RESOLVED'
    if source == 'fuzzy' and (not candidates or candidates[0][1] < CANDIDATE_SCORE_CUTOFF):
        return 'NOT_FOUND'
    return 'NEEDS_CONFIRMATION'

## 4. 정량 비교

후보와 최고 유사도 점수, 상태를 직접 확인한다. `NOT_FOUND`는 기업이 없다는 절대적 사실이 아니라, 현재 후보 생성 기준에서 신뢰할 만한 후보를 제시하지 않는다는 보수적 판정이다.

In [8]:
WARMUP_ITERATIONS = 2
MEASURE_ITERATIONS = 10

def candidate_view(record, score):
    stock_code = record['stock_code'].strip()
    return {
        '법인명': record['corp_name'],
        'corp_code': record['corp_code'],
        '상장 상태': '상장' if stock_code else '비상장',
        '종목코드': stock_code or None,
        '점수': round(score, 1),
    }

def evaluate_policy(policy_name, policy_function):
    rows = []
    latencies = []
    for case in TEST_CASES:
        for _ in range(WARMUP_ITERATIONS):
            candidates, source = find_candidates(case['질문 기업명'])
            policy_function(candidates, source)

        case_latencies = []
        for _ in range(MEASURE_ITERATIONS):
            started_at = time.perf_counter()
            candidates, source = find_candidates(case['질문 기업명'])
            status = policy_function(candidates, source)
            case_latencies.append((time.perf_counter() - started_at) * 1_000)

        rows.append({
            **case,
            '정책': policy_name,
            '후보 출처': source,
            '최고 점수': round(candidates[0][1], 1) if candidates else None,
            '상태': status,
            '후보': [candidate_view(record, score) for record, score in candidates],
        })
        latencies.extend(case_latencies)

    exists_rows = [row for row in rows if row['대상 존재']]
    missing_rows = [row for row in rows if not row['대상 존재']]
    summary = {
        '정책': policy_name,
        'Not-found Recall': sum(row['상태'] == 'NOT_FOUND' for row in missing_rows) / len(missing_rows),
        'False Not-found Rate': sum(row['상태'] == 'NOT_FOUND' for row in exists_rows) / len(exists_rows),
        'Clarification Rate': sum(row['상태'] == 'NEEDS_CONFIRMATION' for row in rows) / len(rows),
        'p50 지연시간(ms)': median(latencies),
        'p95 지연시간(ms)': sorted(latencies)[ceil(len(latencies) * 0.95) - 1],
    }
    return rows, summary

no_cutoff_rows, no_cutoff_summary = evaluate_policy('A. cutoff 없음', no_cutoff_policy)
cutoff_rows, cutoff_summary = evaluate_policy('B. cutoff 75점', cutoff_policy)

for row in no_cutoff_rows + cutoff_rows:
    print({key: row[key] for key in ['정책', '유형', '질문 기업명', '대상 존재', '후보 출처', '최고 점수', '상태', '후보']})

print(no_cutoff_summary)
print(cutoff_summary)

{'정책': 'A. cutoff 없음', '유형': '정확 법인명', '질문 기업명': '삼성전자', '대상 존재': True, '후보 출처': 'exact', '최고 점수': 100.0, '상태': 'RESOLVED', '후보': [{'법인명': '삼성전자', 'corp_code': '00126380', '상장 상태': '상장', '종목코드': '005930', '점수': 100.0}]}
{'정책': 'A. cutoff 없음', '유형': '영문 대소문자', '질문 기업명': 'naver', '대상 존재': True, '후보 출처': 'exact', '최고 점수': 100.0, '상태': 'RESOLVED', '후보': [{'법인명': 'NAVER', 'corp_code': '00266961', '상장 상태': '상장', '종목코드': '035420', '점수': 100.0}]}
{'정책': 'A. cutoff 없음', '유형': '한 글자 오타', '질문 기업명': '삼성전쟈', '대상 존재': True, '후보 출처': 'fuzzy', '최고 점수': 75.0, '상태': 'NEEDS_CONFIRMATION', '후보': [{'법인명': '삼성전자', 'corp_code': '00126380', '상장 상태': '상장', '종목코드': '005930', '점수': 75.0}, {'법인명': '삼성전기', 'corp_code': '00126371', '상장 상태': '상장', '종목코드': '009150', '점수': 75.0}, {'법인명': '삼성기전', 'corp_code': '01024478', '상장 상태': '비상장', '종목코드': None, '점수': 75.0}]}
{'정책': 'A. cutoff 없음', '유형': '부분 표현', '질문 기업명': '하이닉스', '대상 존재': True, '후보 출처': 'fuzzy', '최고 점수': 85.7, '상태': 'NEEDS_CONFIRMATION', '후보': [{'법인명': '이닉스', 'co

## 해석 기준

- B가 Not-found Recall을 높이면서 False Not-found Rate를 높이지 않으면 cutoff 후보로 채택할 근거가 된다.
- 실제 대상이 있는데 `NOT_FOUND`가 되면, 그 기업은 사용자에게 후보 확인 기회도 주지 못하므로 정확도 우선 원칙에서 중요한 실패다.
- synthetic no-match만으로는 운영 cutoff를 확정할 수 없다. 결과가 좋아도 실제 오타·별칭·상호변경·비상장 사례를 추가한 holdout 평가가 필요하다.